# 02 — Exploratory data analysis

**Mini Project 2.** Works through the SA2 frame: the variables individually, their
distributions, the suppression problem, the relationships against crowding, and
the spatial structure. Then closes by writing down the feature list that
notebooks 03 and 04 consume.

Nothing downstream is committed to until the last cell of this notebook, which
writes `outputs/feature_spec.json`.


## Setup

In [ ]:
# ===========================================================================
# CELL B — administrative and urban/rural geography, for notebook 02
#
# Joins the two shapefiles onto the analysis frame so 02 can map crowding by
# territorial authority and urban/rural, and so those columns are in
# feature_spec.json for notebook 04's feature-set comparison.
#
# Run this BEFORE Cell A, since Cell A reads the categorical column names.
#
# The column names inside the two shapefiles are not knowable from the code
# alone, so this probes for them, prints what it found, and fails loudly rather
# than joining on the wrong column.
# ===========================================================================

import geopandas as gpd
import pandas as pd
import numpy as np

NZTM = mp2.NZTM


def find_column(gdf, candidates, what):
    """
    Return the first column of gdf matching any candidate, else any column whose
    name merely contains one. Prints the choice so the assumption is visible.
    """
    exact = [c for c in gdf.columns if c in candidates]
    if exact:
        print(f"  {what:22s} -> {exact[0]!r}  (exact)")
        return exact[0]
    loose = [c for c in gdf.columns
             if any(str(cand).lower() in str(c).lower() for cand in candidates)]
    if loose:
        print(f"  {what:22s} -> {loose[0]!r}  (matched on substring)")
        return loose[0]
    raise KeyError(
        f"No column for {what}. Looked for {candidates}. "
        f"Available columns: {list(gdf.columns)}"
    )


def sa2_code_from(gdf, candidates=("SA22023_V1_00", "SA22023_V1", "SA2",
                                   "SA2_code", "SA22023_CODE")):
    """Pull a 7-digit SA2 code out of whichever id column the shapefile carries."""
    col = find_column(gdf, candidates, "SA2 code")
    s = gdf[col].astype(str).str.extract(r"(\d{4,8})", expand=False)
    return s.str.zfill(7)


# ---------------------------------------------------------------------------
# 1. Territorial authority
# ---------------------------------------------------------------------------
print("Territorial authority shapefile")

ta_gdf = gpd.read_file(DATA / "territorial-authority-2023-generalised.shp")
print(f"  {len(ta_gdf):,} features, CRS {ta_gdf.crs}")

ta_code_col = find_column(ta_gdf, ("TA2023_V1_00", "TA2023_V1_", "TA_CODE",
                                   "Territorial_Authority_Code"), "TA code")
ta_name_col = find_column(ta_gdf, ("TA2023_V1_00_NAME", "TA2023_V_1",
                                   "TA_NAME", "Territorial_Authority_Name"), "TA name")

ta_lookup = (ta_gdf[[ta_code_col, ta_name_col]]
             .rename(columns={ta_code_col: "territorial_authority_code",
                              ta_name_col: "territorial_authority"})
             .drop_duplicates("territorial_authority_code"))

# ---------------------------------------------------------------------------
# 2. Urban / rural
# ---------------------------------------------------------------------------
print()
print("Urban / rural shapefile")

ur_gdf = gpd.read_file(DATA / "urban-rural-2026.shp")
print(f"  {len(ur_gdf):,} features, CRS {ur_gdf.crs}")

ur_name_col = find_column(ur_gdf, ("UR2026_V_2", "UR2026_V1_00_NAME",
                                   "UR_NAME", "Urban_Rural", "UR2026_V_1"), "UR class name")

# ---------------------------------------------------------------------------
# 3. Attach both by spatial join
# ---------------------------------------------------------------------------
# SA2 sits inside TA (a nested hierarchy, so a centroid join is safe), but it
# does NOT sit inside urban/rural — urban areas and rural areas tessellate the
# country independently of SA2 boundaries. So:
#
#   * TA: representative point inside the SA2, then a within-join.
#   * Urban/rural: the same, but reported as a MAJORITY classification, with
#     the share of the SA2's area in the dominant class kept as a confidence
#     column. An SA2 straddling a town boundary is a real case, not an error.

sa2_gdf = gpd.read_file(DATA / "statistical-area-2-2023-generalised.shp")
sa2_gdf["sa2_code"] = sa2_code_from(sa2_gdf)
sa2_gdf = sa2_gdf[sa2_gdf.geometry.notna() & ~sa2_gdf.geometry.is_empty].copy()

if sa2_gdf.crs is None:
    raise ValueError("SA2 boundary file has no CRS — set it before joining.")
if sa2_gdf.crs.to_epsg() != NZTM:
    sa2_gdf = sa2_gdf.to_crs(epsg=NZTM)
    ta_gdf = ta_gdf.to_crs(epsg=NZTM)
    ur_gdf = ur_gdf.to_crs(epsg=NZTM)

sa2_gdf = sa2_gdf[["sa2_code", "geometry"]].drop_duplicates("sa2_code")
sa2_gdf["_point"] = sa2_gdf.representative_point()

# --- TA by representative point
ta_pts = gpd.GeoDataFrame(sa2_gdf[["sa2_code"]], geometry=sa2_gdf["_point"], crs=NZTM)
ta_joined = gpd.sjoin(ta_pts, ta_gdf[[ta_code_col, ta_name_col, "geometry"]],
                      how="left", predicate="within")

# --- urban/rural: area-weighted overlap, then majority
inter = gpd.overlay(sa2_gdf[["sa2_code", "geometry"]],
                    ur_gdf[[ur_name_col, "geometry"]],
                    how="intersection", keep_geom_type=True)
inter["_overlap_km2"] = inter.geometry.area / 1e6

overlap = (inter.groupby(["sa2_code", ur_name_col])["_overlap_km2"]
           .sum().reset_index())
overlap["_sa2_total_km2"] = overlap.groupby("sa2_code")["_overlap_km2"].transform("sum")
overlap["_share"] = overlap["_overlap_km2"] / overlap["_sa2_total_km2"]

dominant = (overlap.sort_values("_share", ascending=False)
            .drop_duplicates("sa2_code")
            .rename(columns={ur_name_col: "urban_rural",
                             "_share": "urban_rural_dominant_share"})
            [["sa2_code", "urban_rural", "urban_rural_dominant_share"]])

# ---------------------------------------------------------------------------
# 4. Merge onto the analysis frame
# ---------------------------------------------------------------------------
before = len(analysis)
analysis = (analysis
      .merge(ta_joined[["sa2_code", ta_name_col]]
             .rename(columns={ta_name_col: "territorial_authority"}),
             on="sa2_code", how="left")
      .merge(dominant, on="sa2_code", how="left")
)
analysis = analysis.loc[:, ~analysis.columns.duplicated()]

print()
print(f"frame rows           : {before:,} -> {len(analysis):,}")
print(f"territorial_authority: {analysis['territorial_authority'].nunique()} distinct, {analysis['territorial_authority'].isna().sum()} unmatched")
print(f"urban_rural          : {analysis['urban_rural'].nunique()} distinct, {analysis['urban_rural'].isna().sum()} unmatched")
print()
print(analysis["urban_rural"].value_counts(dropna=False).to_string())
print()
straddling = (analysis["urban_rural_dominant_share"] < 0.6).sum()
print(f"SA2s where no single urban/rural class holds 60% of the area: {straddling}")
print("(those are boundary cases — the majority label is still the best single")
print(" label, but the share column lets you see how mixed they are)")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

df = pd.read_csv(mp2.OUT / "analysis_frame.csv")
print(f"{len(df):,} SA2s loaded, {df.shape[1]} columns")

### Threshold: top quartile of the SA2 crowding rate

The threshold is computed in notebook 01 and read back from `frame_meta.json`,
so every notebook uses the same one. Chosen over an absolute threshold because a
national-average cut (~6%) would leave the positive class too small for recall or
PR-AUC to mean anything, and because the client ranks areas for need assessment
rather than labelling them.


In [ ]:
meta = json.loads((mp2.OUT / "frame_meta.json").read_text())
threshold = meta["top_quartile_threshold"]

print("threshold      :", round(threshold, 4))
print("positive class :", meta["positive_class"], "of", meta["sa2_with_rate"], "SA2s with a rate")
print("base rate      :", f"{meta['base_rate']:.2%}")
print()
print("predicting the majority class every time scores "
      f"{max(meta['base_rate'], 1 - meta['base_rate']):.2%} accuracy "
      "and 0.00 recall on the crowded class")

## The ranking is noisy at the top

The highest crowding rates in the frame belong to the smallest areas. An SA2 with
3 households where 3 are crowded scores 100%; move one household and it is 67%.
Treating SA2s as equally-weighted units therefore biases any ranking toward tiny
areas. This is a limitation to document, and it needs a size floor before the
ranking is defensible.


In [ ]:
print("most crowded SA2s, with the denominator beside them:")
print(df.nlargest(12, "crowding_rate")[
    ["sa2_name", "crowded_n", "crowding_base", "crowding_rate"]
].to_string(index=False))
print()
for cutoff in (20, 50, 100):
    small = df["crowding_base"] < cutoff
    print(f"SA2s under {cutoff:>3} households: {small.sum():>3}  "
          f"rate range {df.loc[small, 'crowding_rate'].min():.3f} to "
          f"{df.loc[small, 'crowding_rate'].max():.3f}")

## What crowding is in this data

Stats NZ measures household crowding with the Canadian National Occupancy
Standard (CNOS): the number of bedrooms a household needs, given its composition.
The index is published as seven categories plus two totals.

**Collapse.** `crowded` = one or more bedrooms needed, which equals the two
"bedrooms needed" categories summed, to within base-3 rounding. Validated in
notebook 01's reconciliation check.

**Denominator.** `total_stated` — households that gave a usable answer. Dividing
by `total` would include non-response and understate the rate wherever
non-response is higher.


In [ ]:
crowding = [c for c in analysis.columns if c.startswith("household_crowding_index")]

counts  = analysis[crowding].sum()
stated  = counts["household_crowding_index_total_stated"]

levels = pd.DataFrame({
    "households": counts.round(0),
    "percent_of_stated": (counts / stated * 100).round(2),
})
print(levels.to_string())
print()
print(f"total stated : {stated:,.0f}")
print(f"total        : {counts['household_crowding_index_total']:,.0f}")


In [ ]:
print("collapsed target: crowded (one or more bedrooms needed)")
print(f"  crowded households     : {counts['household_crowding_index_crowded']:>10,.0f}")
print(f"  of stated              : {counts['household_crowding_index_crowded']/stated:>10.2%}")
print(f"  not crowded households : {counts['household_crowding_index_not_crowded']:>10,.0f}")


## Suppression and sentinels

Stats NZ writes two numeric sentinels into the measure columns: `-999` for a
suppressed cell and `-997` for a category not collected or not applicable.

`-999` was caught by notebook 01's reconciliation check. `-997` was caught here,
in a column whose every value was the sentinel — which is why a missing-value
count returned zero for it. `mp2.read_measure` now masks anything at or below
`-990`, so both are handled before the frame is written.

The distribution table below is the check that it worked.


In [ ]:
#now I need to re run the distribution code to without the -997 numbers
numeric_features = [c for c in analysis.columns
                    if c.startswith(("tenure_of_household", "total_household_income",
                                     "number_of_motor_vehicles", "household_composition",
                                     "sector_of_landlord",
                                     "access_to_telecommunication_systems",
                                     "number_of_usual_residents"))]

summary = (analysis[numeric_features].describe().T[["count", "mean", "std", "min", "50%", "max"]])
summary["missing"] = analysis[numeric_features].isna().sum()
print(summary.round(2).to_string())


### Suppression is not missing-at-random

The README claims suppression concentrates in small areas. This measures it, so
the claim has a number behind it. The 112 SA2s that lose the crowding rate
entirely are excluded from the model, and they are systematically the small ones
— which is the model's blind spot, stated rather than glossed.


In [ ]:
# any Stats NZ sentinel: -997 and -999 and anything else in that range
numeric = [c for c in analysis.columns
           if c not in ("sa2_code", "sa2_name") and analysis[c].dtype == "float64"]
analysis[numeric] = analysis[numeric].mask(analysis[numeric] <= -990)
analysis = analysis.copy()

print("columns still holding negatives:",
      int((analysis[numeric] < 0).any().sum()))
print("cells now missing:", int(analysis[numeric].isna().sum().sum()))


In [ ]:
#check that there are no other values like this
all_missing = [c for c in numeric if analysis[c].isna().all()]
print(f"{len(all_missing)} columns are entirely missing and will be dropped:")
for c in all_missing:
    print("   ", c)

analysis = analysis.drop(columns=all_missing)
print()
print("frame now:", analysis.shape)


### A size floor

`meets_min_size` flags areas with enough households for the rate to be stable.
It is written into the frame so notebooks 03–05 can apply it or argue against it
in one place. The coverage line matters: dropping tiny areas costs very few
households.


In [ ]:
small = analysis["crowding_base"] < 20
print("SA2s under 20 households:", small.sum())
print("their crowding rate range:", 
      f"{analysis.loc[small,'crowding_rate'].min():.3f} to {analysis.loc[small,'crowding_rate'].max():.3f}")
print()
print("SA2s under 50 households:", (analysis['crowding_base'] < 50).sum())
print("SA2s under 100 households:", (analysis['crowding_base'] < 100).sum())
print()
print(analysis.nsmallest(12, "crowding_base")[
    ["sa2_name", "crowded_count", "crowding_base", "crowding_rate"]
].to_string(index=False))


## Relationships against crowding

Three bivariate checks, one point per SA2. These are the evidence that a
multivariate model has signal to find — and the R² values are deliberately
reported, because they are low and that is the argument for using the profile
together rather than any one variable alone.


In [ ]:
# Check your original dataframe for income columns (change 'analysis' if your raw dataframe is named differently)
income_cols_to_restore = [c for c in analysis.columns if 'income' in c]
print("Found income columns to restore:")
for c in income_cols_to_restore:
    print(f" - {c}")

# Merge them back into the joined frame using sa2_code as the key
if 'sa2_code' in df.columns and 'sa2_code' in analysis.columns:
    cols_to_add = ['sa2_code'] + income_cols_to_restore
    df = df.merge(analysis[cols_to_add], on='sa2_code', how='left')
    print(f"Successfully restored income columns! New shape: {df.shape}")

In [ ]:
income_col = "total_household_income_median"
crowding_col = "crowding_rate"

rows = []
fits = {}

for col in ["total_household_income_median",
            "weekly_rent_paid_by_household_under_200",
            "number_of_motor_vehicles_no_motor_vehicle"]:
    if col not in df.columns:
        print("missing column, skipped:", col)
        continue
    x, y = df[col], df[crowding_col]
    mask = x.notna() & y.notna()
    slope, intercept = np.polyfit(x[mask], y[mask], 1)
    r2 = np.corrcoef(x[mask], y[mask])[0, 1] ** 2
    fits[col] = (slope, intercept, r2)
    rows.append({"feature": col, "n": int(mask.sum()),
                 "slope": f"{slope:.2e}", "r_squared": round(r2, 4)})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")

for ax, (col, (slope, intercept, r2)) in zip(axes, fits.items()):
    x, y = df[col], df[crowding_col]
    mask = x.notna() & y.notna()
    xc, yc = x[mask], y[mask]
    ax.scatter(xc, yc, s=12, alpha=0.4, color="#2F5D9E")
    ax.plot(xc, slope * xc + intercept, color="#D9534F", linewidth=2,
            label=f"slope {slope:.2e}\nR² {r2:.4f}")
    ax.set_xlabel(col.replace("_", " "), fontsize=9)
    ax.set_ylabel("Crowding rate")
    ax.legend(fontsize=8, frameon=True)
    ax.grid(True, linestyle="--", alpha=0.3)

plt.show()

## Spatial structure

Two maps, both from the join in the setup cell: crowding rate by SA2, and the
urban/rural classification the join attached. The premise of the whole project is
that crowding has spatial structure, so it should be visible here.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# 1. Ensure keys match data types and project to NZTM2000
df["sa2_code"] = df["sa2_code"].astype(str)
gdf["SA22023_V1"] = gdf["SA22023_V1"].astype(str)

if gdf.crs != "EPSG:2193":
    gdf = gdf.to_crs("EPSG:2193")

# 2. Left join so all suppressed/missing shapefile rows are retained
df = gdf.merge(
    df, left_on="SA22023_V1", right_on="sa2_code", how="left"
)

# 3. Create the figure
fig, ax = plt.subplots(figsize=(9, 11), layout="constrained")

# Plot main choropleth
df.plot(
    column="crowding_rate",
    cmap="YlOrRd",
    legend=False,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "white",
        "linewidth": 0.2,
    },
    edgecolor="grey",
    linewidth=0.08,
    ax=ax,
)

ax.set_title(
    "Household Crowding Rate by Statistical Area 2 (SA2)\n2023 Census (NZTM2000)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_axis_off()

# ==========================================
# 4. ADD NORTH ARROW (Top Right)
# ==========================================
ax.annotate(
    "N",
    xy=(0.93, 0.93),
    xycoords="axes fraction",
    fontsize=16,
    fontweight="bold",
    ha="center",
    va="center",
)
ax.annotate(
    "",
    xy=(0.93, 0.91),
    xycoords="axes fraction",
    xytext=(0.93, 0.86),
    textcoords="axes fraction",
    arrowprops=dict(facecolor="black", width=1.5, headwidth=6),
)

# ==========================================
# 5. ADD SCALE BAR (Shifted to Bottom Right)
# ==========================================
# Adjusted NZTM meter coordinates for the lower-right area
scale_x, scale_y, scale_length_m = 1750000, 4850000, 200000
ax.plot(
    [scale_x, scale_x + scale_length_m],
    [scale_y, scale_y],
    color="black",
    linewidth=2,
)
ax.plot(
    [scale_x, scale_x], [scale_y - 15000, scale_y + 15000], color="black", linewidth=1.5
)
ax.plot(
    [scale_x + scale_length_m, scale_x + scale_length_m],
    [scale_y - 15000, scale_y + 15000],
    color="black",
    linewidth=1.5,
)
ax.text(
    scale_x + scale_length_m / 2,
    scale_y + 30000,
    "200 km",
    ha="center",
    va="bottom",
    fontsize=9,
)

# ==========================================
# 6. ADD COMPREHENSIVE LEGEND (Bottom Right)
# ==========================================
legend_handles = [
    mpatches.Patch(color="#ffffb2", ec="grey", label="Low crowding (< 3%)"),
    mpatches.Patch(color="#fecc5c", ec="grey", label="Moderate crowding"),
    mpatches.Patch(color="#fd8d3c", ec="grey", label="Elevated crowding"),
    mpatches.Patch(color="#e31a1c", ec="grey", label="High crowding (> 30%)"),
    mpatches.Patch(
        color="lightgrey",
        ec="white",
        label="Suppressed / No data (154 SA2s)",
    ),
]

ax.legend(
    handles=legend_handles,
    title="Household Crowding Rate\n(CNOS Standard)",
    loc="lower right",
    frameon=True,
    facecolor="white",
    edgecolor="none",
    fontsize=9,
    title_fontsize=10,
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

# 1. Quick check: Ensure your boundary file has the Urban/Rural column loaded
# (Replace 'path_to_ur_shapefile.shp' with your actual loaded variable name if it's already in memory)
# If urban_rural_gdf is already loaded, you can skip reading it again:
# df = gdf.merge(analysis, left_on="SA22023_V1", right_on="sa2_code", how="left")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 2. Plot Urban / Rural Classification
if "UR2026_V_2" in df.columns:
    df.plot(
        column="UR2026_V_2",
        categorical=True,
        cmap="Set2",
        legend=True,
        legend_kwds={"bbox_to_anchor": (1.05, 1), "loc": "upper left"},
        missing_kwds={"color": "lightgrey", "label": "Missing"},
        ax=axes[0],
    )
    axes[0].set_title("Urban / Rural Classification", fontsize=12, fontweight="bold")
else:
    axes[0].text(0.5, 0.5, "UR2026_V_2 column not found in df", 
                 horizontalalignment='center', verticalalignment='center', transform=axes[0].transAxes)
axes[0].set_axis_off()

# 3. Plot Census Crowding Rate
if "crowding_rate" in df.columns:
    df.plot(
        column="crowding_rate",
        cmap="YlOrRd",
        legend=True,
        legend_kwds={"label": "Crowding Rate", "orientation": "horizontal"},
        missing_kwds={"color": "lightgrey", "label": "Suppressed / Missing"},
        ax=axes[1],
    )
    axes[1].set_title("Census Household Crowding Rate", fontsize=12, fontweight="bold")
else:
    axes[1].text(0.5, 0.5, "crowding_rate column not found", 
                 horizontalalignment='center', verticalalignment='center', transform=axes[1].transAxes)
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

The number to read off the join: how many SA2s have no single
urban/rural class holding 60% of their area. Those are genuine boundary cases,
and the majority label is the best single label for them but not a strong claim.


In [ ]:
import matplotlib.pyplot as plt

# 1. Inspect unique values and non-null counts in your dataframe
print("Unique values in UR2026_V_2:", df["UR2026_V_2"].unique())
print("Non-null count:", df["UR2026_V_2"].notna().sum())

# 2. Plot just the Urban / Rural classification cleanly
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Convert to string and filter out any missing geometries or NaNs just for this plot
plot_df = df.dropna(subset=["UR2026_V_2"])

plot_df.plot(
    column="UR2026_V_2",
    categorical=True,
    cmap="Set2",
    legend=True,
    legend_kwds={"bbox_to_anchor": (1.05, 1), "loc": "upper left"},
    ax=ax,
)

ax.set_title("Urban / Rural Classification (Stats NZ)", fontsize=14, fontweight="bold")
ax.set_axis_off()

plt.tight_layout()
plt.show()

## The feature list

Three deliberate exclusions, each with a reason the README can carry:

**The target.** Every `household_crowding_index_*` column, plus the derived
`crowding_rate`, `crowded_n`, `crowding_base` and `crowded_top_quartile`.

**The target restated.** Household composition and household size. Crowding is
defined as composition against a bedroom requirement, so these are the outcome in
another form. The brief asks whether the area's *profile* can stand in for a
direct measurement — feeding the measurement itself defeats that. Held out of the
primary list, and available for a sensitivity run in notebook 04.

**Denominators and residuals.** Every `_total`, `_total_stated` and `_median` is
a scale variable or is derived from the brackets it accompanies; `_not_stated` and
`_not_elsewhere_included` are non-response and residual buckets, not profile.
Used to build the shares, then dropped.

**Counts become shares.** A raw bracket count is large in any big suburb
regardless of composition. Dividing each bracket by its own stated total turns it
into a share, which is what makes the clustering reflect composition rather than
area size — the same problem the size floor addresses above.


In [ ]:
# ===========================================================================
# CELL A — feature selection, for the END of notebook 02
#
# Replaces the commented-out "TODO: persist the feature choices" block.
# Writes outputs/feature_spec.json in the shape notebooks 03/04/05 already read:
#
#     spec["numeric"]  spec["categorical"]  spec["target"]  spec["leaky_or_id"]
#
# Three decisions are encoded here, each justified in the markdown above it.
# ===========================================================================

import json
import numpy as np
import pandas as pd

# Confirm the sentinel fix actually landed before building features on it.
mp2.assert_no_sentinels(df)

# ---------------------------------------------------------------------------
# 1. The three exclusions
# ---------------------------------------------------------------------------

# (a) The target itself. Every crowding index column, plus the four derived
#     columns 01 wrote. Including any of these would leak the answer.
TARGET_PREFIX = "household_crowding_index_"
DERIVED_TARGET = {"crowded_count", "crowding_base", "crowding_rate",
                  "crowded_top_quartile", "crowded_n"}

# (b) The outcome restated. Crowding is household composition measured against
#     a bedroom requirement, so household size and composition ARE the target
#     expressed differently. The brief asks whether the area's *profile* can
#     stand in for a direct measurement; feeding in the measurement defeats it.
#     Held out of the primary feature set and used once as a sensitivity run.
DEFINITION_PREFIXES = (
    "household_composition_",
    "number_of_usual_residents_in_household_",
    "count_of_households_in_occupied_private_dwellings",
)

# ---------------------------------------------------------------------------
# 2. Bracket families, converted to shares of their own stated total
# ---------------------------------------------------------------------------
# A raw count bracket is large in any big suburb regardless of composition.
# Dividing by the stated total turns "how many households" into "what share of
# households", which is what makes the clustering reflect composition rather
# than area size. The denominator is used and then dropped.

SHARE_FAMILIES = {
    "total_household_income": "total_household_income_total_stated",
    "weekly_rent_paid_by_household": "weekly_rent_paid_by_household_total_stated",
    "sector_of_landlord": "sector_of_landlord_total_stated",
    "tenure_of_household": "tenure_of_household_total_stated",
    "number_of_motor_vehicles": "number_of_motor_vehicles_total_stated",
    "access_to_telecommunication_systems_total_responses":
        "access_to_telecommunication_systems_total_responses_total_stated",
}

# Dropped after use. Denominators are scale variables; medians are derived from
# the same brackets; "not stated" and "not elsewhere included" are non-response
# and residual buckets, not socio-economic profile.
DROP_SUFFIXES = ("_total_stated", "_total", "_median",
                 "_not_stated", "_not_elsewhere_included", "_unidentifiable",
                 "_average")


def _is_denominator(col):
    return col in SHARE_FAMILIES.values()


def _share_denominator(col):
    """The stated total this bracket belongs to, or None if it is not a bracket."""
    if _is_denominator(col):
        return None
    for prefix, denom in SHARE_FAMILIES.items():
        if col.startswith(prefix + "_"):
            return denom
    return None


def _excluded(col):
    if col.startswith(TARGET_PREFIX) or col in DERIVED_TARGET:
        return "target"
    if any(col.startswith(p) for p in DEFINITION_PREFIXES):
        return "definition"
    return None


# ---------------------------------------------------------------------------
# 3. Classify every column in the frame
# ---------------------------------------------------------------------------

measure_cols = [c for c in df.columns
                if c not in ("sa2_code", "sa2_name")
                and pd.api.types.is_numeric_dtype(df[c])]

rows = []
for c in measure_cols:
    why = _excluded(c)
    if why:
        rows.append({"column": c, "role": why})
        continue
    denom = _share_denominator(c)
    if denom is not None:
        rows.append({"column": c, "role": "share", "denominator": denom})
    elif c.endswith(DROP_SUFFIXES):
        rows.append({"column": c, "role": "denominator"})
    else:
        rows.append({"column": c, "role": "keep"})

roles = pd.DataFrame(rows)
print(roles["role"].value_counts().to_string())
print()

# ---------------------------------------------------------------------------
# 4. Build the share columns, then select the feature list
# ---------------------------------------------------------------------------

share_rows = roles[roles["role"] == "share"]
for _, r in share_rows.iterrows():
    col, denom = r["column"], r["denominator"]
    if denom not in df.columns:
        raise KeyError(f"{col!r} expects denominator {denom!r}, which is not in the frame")
    df[f"{col}_share"] = (df[col] / df[denom].replace(0, np.nan)).astype("float64")

NUMERIC = (list(roles.loc[roles["role"] == "keep", "column"])
           + [f"{c}_share" for c in share_rows["column"]])

# ---------------------------------------------------------------------------
# 5. Categoricals — attached by the spatial cell below, or already present
# ---------------------------------------------------------------------------

CATEGORICAL = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])
               and c not in ("sa2_name",)]

# ---------------------------------------------------------------------------
# 6. Verify before writing: nothing leaky, nothing degenerate
# ---------------------------------------------------------------------------

leaked = [c for c in NUMERIC + CATEGORICAL
          if c.startswith(TARGET_PREFIX) or c in DERIVED_TARGET
          or any(c.startswith(p) for p in DEFINITION_PREFIXES)]
assert not leaked, f"excluded columns reached the feature list: {leaked}"

# A share column that is entirely missing is a column to drop, not to impute.
all_missing = [c for c in NUMERIC if df[c].isna().all()]
assert not all_missing, f"feature columns with no observed values: {all_missing}"

# The shares of a family should sum to ~1 where the denominator is present.
for prefix, denom in SHARE_FAMILIES.items():
    fam = [c for c in NUMERIC if c.startswith(prefix + "_") and c.endswith("_share")]
    if not fam:
        continue
    total = df.loc[df[denom].notna(), fam].sum(axis=1, min_count=1)
    print(f"{prefix:58s} n={len(fam):2d}  "
          f"share sum  median {total.median():.3f}  "
          f"p05 {total.quantile(0.05):.3f}  p95 {total.quantile(0.95):.3f}")

# ---------------------------------------------------------------------------
# 7. Write the spec
# ---------------------------------------------------------------------------

spec = {
    "numeric": NUMERIC,
    "categorical": CATEGORICAL,
    "target": "crowded_top_quartile",
    "leaky_or_id": sorted(set(
        [c for c in df.columns if c.startswith(TARGET_PREFIX)]
        + list(DERIVED_TARGET)
        + ["sa2_code", "sa2_name", "geometry"]
        + [c for c in df.columns if any(c.startswith(p) for p in DEFINITION_PREFIXES)]
    )),
    "definition_only": [c for c in df.columns
                        if any(c.startswith(p) for p in DEFINITION_PREFIXES)],
    "share_denominators": SHARE_FAMILIES,
    "target_cohort": "crowded_top_quartile",
    "n_rows": int(len(df)),
    "n_numeric": len(NUMERIC),
    "n_categorical": len(CATEGORICAL),
}

(mp2.OUT / "feature_spec.json").write_text(json.dumps(spec, indent=2))

print()
print(f"numeric features     : {len(NUMERIC)}")
print(f"categorical features : {len(CATEGORICAL)}  {CATEGORICAL}")
print(f"excluded as target   : {len([c for c in spec['leaky_or_id'] if c.startswith(TARGET_PREFIX)])}")
print(f"excluded as definition: {len(spec['definition_only'])}")
print()
print("written:", mp2.OUT / "feature_spec.json")
print()
print("numeric feature list:")
for i, c in enumerate(NUMERIC, 1):
    print(f"  {i:2d}. {c}")

In [ ]:
print()
print("categorical list (from the spatial join in setup):")
print(spec["categorical"])